In [ ]:
# Ensure gradalg is importable when this notebook is opened
# directly (not via pytest). Walks up from the notebook's
# directory to the repo root and prepends it to sys.path if
# gradalg isn't already installed into this kernel.
try:
    import gradalg  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "gradalg" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import gradalg  # noqa: F401

# 01 — İlk Adımlar

Bu notebook [01_first_steps.md](01_first_steps.md) markdown'ının çalıştırılabilir sürümüdür. `gradalg`'de sembolik ifade kurma, özellik atama ve sadeleştirmenin temelleri.

## Semboller ve temel inşa

`Symbol(name)` atomik bir ifade üretir; `Integer(n)` literal sabit; `+`, `*`, `-` arka planda `Sum`, `Product`, `Neg` inşa eder.

In [ ]:
from gradalg.core.expr import Symbol, Integer, Sum, Product, Neg

x, y, z = Symbol("x"), Symbol("y"), Symbol("z")

expr = x + y - z
assert expr == Sum(x, y, Neg(z))
print(expr)

In [ ]:
mult = 2 * x
assert mult == Product(Integer(2), x)
print(mult)

## Özellik atama (PropertyRegistry)

Özellikler — derece, skalerlik, graded antisymmetry, … — semboller üzerine *dışsal* olarak ilan edilir. Bu, aynı sembolün farklı bağlamlarda (farklı derecelerde, farklı cebirlerde) yeniden kullanılmasına izin verir.

In [ ]:
from gradalg.core.properties import Graded, Scalar
from gradalg.core.registry import PropertyRegistry

reg = PropertyRegistry()
reg.declare(x, Scalar())
reg.declare(y, Scalar())
reg.declare(z, Graded(degree=1))  # z bir 1-form gibi davranır

### Role-driven kısayollar

Sık tekrarlanan desenler — fonksiyon, vektör alanı, form, bivector — için `gradalg.library.declarations` altında `Functions`, `VectorFields`, `Forms`, `Bivector` yardımcıları var. Her biri `Symbol(...)` + uygun `reg.declare(...)` çağrısını tek satıra indirir.

In [ ]:
from gradalg import Functions, VectorFields, Forms, Bivector

reg2 = PropertyRegistry()
f, g = Functions("f g", registry=reg2)
X, Y = VectorFields("X Y", registry=reg2)
alpha, beta = Forms("α β", degree=1, registry=reg2)
pi = Bivector("π", registry=reg2)
print(f, g, X, Y, alpha, beta, pi)

## `simplify`: canonical forma indirme

`simplify(expr, registry)` pipeline'ı: flatten → canonicalize → distribute → flatten → sort_product → collect_terms. Registry verilirse kayıtlı özellikler (commutativity, derece) işleme alınır.

In [ ]:
from gradalg.algorithms.simplify import simplify

assert simplify(x + x - x) == x
assert simplify(Product(Integer(2), x, Integer(3)), reg) \
    == Product(Integer(6), x)
# Scalar iki sembol alfabetik sıraya alınır:
assert simplify(Product(y, x), reg) == Product(x, y)
print('simplify checks passed')

## Görselleştirme

`display` katmanı `to_ascii`, `to_latex` ve (rich yüklüyse) renkli terminal ağacı verir.

In [ ]:
from gradalg.display import to_ascii, to_latex

e = x + y - z
print('ascii:', to_ascii(e))
print('latex:', to_latex(e))

## Sonraki adım

Bir bracket üstüne Jacobi ispatı: [02_jacobi_identity.md](02_jacobi_identity.md).